<a href="https://colab.research.google.com/github/gmauricio-toledo/NLP-LCC/blob/main/Notebooks/12-Semantica_Distribucional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Semántica distribucional: similitud y ambigüedad en distintos modelos</h1>

Convertir texto en vectores numéricos es el paso fundamental que permite aplicar álgebra lineal y geometría a problemas lingüísticos. Sin embargo, no todos los embeddings capturan el mismo tipo de significado: un modelo de bolsa de palabras, un embedding estático tipo Word2Vec o un modelo contextual como BERT producen representaciones con propiedades muy distintas. Esta notebook explora esas diferencias de forma ilustrativa a través de dos experimentos: uno que compara cómo distintos modelos miden la similitud entre textos cortos, y otro que examina cómo los embeddings estáticos manejan la polisemia.

# Similitud entre textos

Un mismo par de oraciones puede resultar muy similar o muy distinto dependiendo del modelo que se use para representarlas. En esta sección comparamos cuatro enfoques — BoW, Doc2Vec, Word2Vec promediado y Transformers — sobre un conjunto de textos cortos. El objetivo no es evaluar cuál modelo es mejor, sino observar qué aspectos del significado privilegia cada representación: coincidencia léxica, temática general, o similitud semántica profunda.

Trabajaremos con los siguientes textos:

In [ ]:
texto_1 = "i feel fine today"
texto_2 = "i am doing great this day"
texto_3 = "i feel not fine today"
texto_4 = "the car is blue and new"

corpus = [texto_1, texto_2, texto_3, texto_4]

🔴 ¿Cómo deberían ser las similitudes entre ellos?

## Bag of Words / TF-IDF

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus).todense()
X = np.array(X)
X.shape

In [ ]:
from numpy import linalg

def cosine_similarity(a, b):
    try:
        return np.dot(a,b) / (np.linalg.norm(a) * np.linalg.norm(b))
    except:
        return 0

In [ ]:
for j,text in enumerate(corpus):
    print(f"texto {j}: {text}")
print(50*'-')

for j in range(1, len(corpus)):
    sim = cosine_similarity(X[0], X[j])
    print(f"Similitud entre 0 y {j}: {round(sim,2)}")

## Word2vec

In [ ]:
!pip install gensim

In [ ]:
import gensim

for x in gensim.downloader.info()['models'].keys():
    print(x)

In [ ]:
import gensim.downloader as api

model = api.load("glove-wiki-gigaword-50")

In [ ]:
vectors = []
for text in corpus:
    vector = model.get_mean_vector(text.split())
    vectors.append(vector)

vectors = np.array(vectors)
vectors.shape

In [ ]:
for j, text in enumerate(corpus):
    print(f"texto {j}: {text}")
print(50*'-')

for j in range(1, len(corpus)):
    sim = cosine_similarity(vectors[0], vectors[j])
    print(f"Similitud entre 0 y {j}: {round(sim,2)}")

## Doc2Vec

🛑 Omitimos este modelo porque no tenemos muchos textos para entrenarlo

In [ ]:
!gdown 1kzm3n2PSXyz9_F8j2StWy-sKAOq_ZhOJ

In [ ]:
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import nltk
nltk.download('punkt_tab', quiet=True)

docs = pd.read_csv('IMDB50K_cleaned.csv')['cleaned_review'].values

tagged_data = [TaggedDocument(words=nltk.word_tokenize(doc), tags=[str(i)]) for i, doc in enumerate(docs)]

model = Doc2Vec(vector_size=50, min_count=2, epochs=40, window=5)
model.build_vocab(tagged_data)
model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)

In [ ]:
d2v_vectors = []

for text in corpus:
    vector = model.infer_vector(nltk.word_tokenize(text))
    d2v_vectors.append(vector)

d2v_vectors = np.array(d2v_vectors)
d2v_vectors.shape

In [ ]:
for j, text in enumerate(corpus):
    print(f"texto {j}: {text}")

print(50*'-')

for j in range(1, len(corpus)):
    sim = cosine_similarity(d2v_vectors[0], d2v_vectors[j])
    print(f"Similitud entre 0 y {j}: {round(sim,2)}")

## Sentence Transformers

In [ ]:
from sentence_transformers import SentenceTransformer

# model = SentenceTransformer('all-MiniLM-L6-v2')
model = SentenceTransformer('paraphrase-MiniLM-L3-v2')
st_vectors = model.encode(corpus)

In [ ]:
for j, text in enumerate(corpus):
    print(f"texto {j}: {text}")
print(50*'-')

for j in range(1, len(corpus)):
    sim = cosine_similarity(st_vectors[0], st_vectors[j])
    print(f"Similitud entre 0 y {j}: {round(sim,2)}")

# Polisemia y embeddings estáticos

Los modelos de embeddings estáticos como Word2Vec asignan un único vector a cada palabra, sin importar el contexto en que aparezca. Para palabras con un solo significado dominante esto funciona razonablemente bien, pero para palabras polisémicas el embedding termina siendo un promedio de acepciones distintas. En esta sección examinamos los vecinos más cercanos de algunas palabras polisémicas y monosémicas para ver este efecto directamente, estableciendo la motivación para los modelos contextuales como BERT.

## Leemos un modelo

In [ ]:
!pip install gensim

In [ ]:
import gensim.downloader as api

model = api.load("glove-wiki-gigaword-100")

## Palabras polisémicas

In [ ]:
model.most_similar("bow")

In [ ]:
model.most_similar("mole",topn=20)

In [ ]:
model.most_similar("crane",topn=20)

## Palabras monosémicas

In [ ]:
model.most_similar("good")

In [ ]:
model.most_similar("computer")